# Reference-run workflow: parse → reconcile → verify → fit → verify → optimise

Walks a single saved `.jld2` run through this package's 1st-order pipeline end to end:

1. Load a reference `.jld2` file.
2. Parse and display its cavity / ensemble / control-pulse / signal-pulse settings.
3. Re-run the 1st-order ODE (`:ground` track) and reconcile the **final state** against
   the file's own recorded final state (`|a⟩`, `S_j^z`, `S_j^+`, `S_j^-`, inversion).
4. Re-run the 1st-order ODE (`:equator` track) and compute silencing factor + coherence,
   displayed alongside step 3's inversion.
5. Fit a parameterised (B-spline `CompositePulse`) curve to the recorded control pulse,
   reporting the fit error broken out by envelope, Rabi amplitude, and instantaneous
   transition frequency.
6. Re-run a dual-track (`:ground` + `:equator`) ODE solve on the **fitted** curve and
   display its inversion / silencing / coherence, for direct comparison against steps 3-4.
7. Run the differentiable pulse optimiser (`optimise_composite_pulse`, warm-started from
   the step-5 fit) with **no k-hopping and no multi-k-seeding** — a single `CompositePulse`
   shape, a single seed, one local Adam descent.

**Kernel:** Julia, with this repo as the active project (`Pkg.activate` below). `IJulia`
must be installed in that environment.


In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))

using InhomogeneousSpinCavityDynamics
using Printf


## 1. Reference file

Point `JLD2_PATH` at any run written by `save_run_data` (must contain `SIM_SETTING`,
`SYSTEM_CONFIG`, `PULSE_CONFIG`, and the saved trajectory `a_sol`/`Σp_sol`/`Σz_sol`).
`N_SIGNAL` is how many *leading* entries of `PULSE_CONFIG` are the fixed signal pulse
(the rest are the control pulse) — see [`split_signal_control`](../src/jld2_pulse_loader.jl).


In [ ]:
JLD2_PATH = joinpath(@__DIR__, "..", "data", "data_1st_order", "3ARP_pi_gstd_1em06Hz.jld2")
N_SIGNAL  = 1

isfile(JLD2_PATH) || error("JLD2 file not found: $JLD2_PATH")
println("JLD2_PATH = $JLD2_PATH")


In [ ]:
data = load_jld2_run(JLD2_PATH)
CONFIG = build_full_config(data.SIM_SETTING, data.SYSTEM_CONFIG)
d = prepare_derived(CONFIG)
signal_cfg, control_cfg = split_signal_control(data.PULSE_CONFIG; n_signal=N_SIGNAL)

# Reused for every re-solve below so all of steps 3/4/6 are compared on equal footing,
# and default to the SAME tolerances the reference run itself was recorded at.
RELTOL = data.SIM_SETTING.reltol
ABSTOL = data.SIM_SETTING.abstol

println("Loaded $JLD2_PATH")
println("M = $(d.M)  (M_delta=$(d.M_delta) x M_g=$(d.M_g))   reltol=$RELTOL  abstol=$ABSTOL")


## 2. Parsed settings — cavity / ensemble / control pulse / signal pulse

In [ ]:
println("=== Cavity ===")
@printf("  kappa_e (external) = %.6g\n", d.kappa_e)
@printf("  kappa_i (internal) = %.6g\n", d.kappa_i)
@printf("  kappa_t (total)    = %.6g\n", d.kappa_t)
@printf("  delta0             = %.6g\n", d.delta0)

println()
println("=== Ensemble ===")
@printf("  C_ens (cooperativity)     = %.6g\n", d.C_ens)
@printf("  N (total spin number)     = %.6g\n", d.N)
@printf("  N_total (binned check)    = %.6g\n", d.N_total)
@printf("  M_delta x M_g -> M        = %d x %d -> %d bins\n", d.M_delta, d.M_g, d.M)
@printf("  FWHM (frequency inhom.)   = %.6g\n", d.FWHM)
@printf("  g_mean / g_std (coupling) = %.6g / %.6g\n", d.g_mean, d.g_std)
@printf("  timespan                  = (%.6g, %.6g) s\n", d.timespan[1], d.timespan[2])

println()
println("=== Signal pulse(s) ($(length(signal_cfg))) ===")
for (i, cfg) in enumerate(signal_cfg)
    println("  [$i] kind=$(cfg.kind)  $cfg")
end

println()
println("=== Control pulse(s) ($(length(control_cfg))) ===")
for (i, cfg) in enumerate(control_cfg)
    println("  [$i] kind=$(cfg.kind)  $cfg")
end


## 3. Reconcile the 1st-order ODE solve against the recorded final state

Re-solves the recorded signal+control drive on the **`:ground`** track and compares the
final state against `data.a_sol[end]` / `data.Σp_sol[end]` / `data.Σz_sol[end]` — the file
only stores the **ensemble-summed** `⟨a⟩`, `Σ S^+ = Σ_j S_j^+`, `Σ S^z = Σ_j S_j^z`, so
those sums (not a per-bin `max` over `M` individual spins) are the finest-grained comparison
available against the recorded trajectory. `S_j^-` is not stored separately: for a physical
state `⟨S^-⟩ = conj(⟨S^+⟩)` exactly, so its error is identical in magnitude to `S^+`'s —
computed explicitly below rather than just asserted.

Uses [`reconcile_against_jld2`](../src/jld2_pulse_loader.jl), which is exactly this check.


In [ ]:
ok, report, _, _ = reconcile_against_jld2(JLD2_PATH; n_signal=N_SIGNAL, verbose=true)

# S^- = conj(S^+) identity, made explicit (not just asserted) for the table below.
signal_E_r = build_E_of_t(signal_cfg)
control_E_r = build_E_of_t(control_cfg)
E_of_t_r(t) = signal_E_r(t) + control_E_r(t)
a_chk, Sp_chk, Sz_chk = run_sim_1st_order_final(E_of_t_r, d; initial_condition=:ground, reltol=RELTOL, abstol=ABSTOL)
Sigma_m_chk = conj(sum(Sp_chk))
Sigma_m_ref = conj(data.Σp_sol[end])
err_m = abs(Sigma_m_chk - Sigma_m_ref)
atol_use = InhomogeneousSpinCavityDynamics._default_final_state_atol(ABSTOL)  # same floor reconcile_against_jld2 used
rel_m = err_m / (abs(Sigma_m_ref) + atol_use)

println()
println("=== Final-state reconciliation (:ground track) ===")
@printf("  |a⟩  abs_err=%.6g  rel_err=%.6g\n", report.err_a, report.rel_a)
@printf("  S^z  abs_err=%.6g  rel_err=%.6g\n", report.err_z, report.rel_z)
@printf("  S^+  abs_err=%.6g  rel_err=%.6g\n", report.err_p, report.rel_p)
@printf("  S^-  abs_err=%.6g  rel_err=%.6g  (= S^+ error, by conj symmetry)\n", err_m, rel_m)
@printf("  inversion = %.6g\n", report.inversion)
println("  status: ", ok ? "PASS" : "FAIL", " (rtol=1e-3)")


## 4. Equatorial-track solve — silencing factor & coherence

Same recorded signal+control drive, `:equator` initial condition this time. Paired with
step 3's `:ground`-track inversion, this gives the same three headline metrics
[`pulse_metrics`](../src/pulse_optimizer2.jl) reports for an optimised pulse — computed
here for the file's **original, unmodified** recorded pulse, as a baseline to compare the
fitted/optimised pulse against in steps 6-7.


In [ ]:
a_eq, Sp_eq, Sz_eq = run_sim_1st_order_final(E_of_t_r, d; initial_condition=:equator, reltol=RELTOL, abstol=ABSTOL)

silencing_ref = InhomogeneousSpinCavityDynamics._weighted_silencing_factor(Sp_eq, d.g_b, d.Nj, Float64)
coherence_ref = InhomogeneousSpinCavityDynamics._weighted_coherence(Sp_eq, d.Nj, Float64)
inversion_ref = report.inversion

println("=== Recorded pulse (original, unmodified) ===")
@printf("  inversion = %.6g\n", inversion_ref)
@printf("  silencing = %.6g\n", silencing_ref)
@printf("  coherence = %.6g\n", coherence_ref)


## 5. Parameterised curve fit of the recorded control pulse

Fits a B-spline [`CompositePulse`](../src/composite_pulse.jl) to `control_cfg`'s own
sampled `(I, Q)` trace via [`fit_composite_pulse_seed_auto`](../src/jld2_pulse_loader.jl)
(`fit_mode=:linear`, the closed-form per-segment weighted least-squares route — cheap even
at high resolution, the same route `optimise_control_pulse_from_jld2` uses). `k` is
auto-detected from the trace itself; `param_budget=60` caps the per-segment spline
coefficient counts to the same default `optimise_control_pulse_from_jld2` itself uses —
left unset, the auto-sizing tracks raw sample density instead (thousands of coefficients
for a densely-sampled trace), which would force step 7's optimiser into many small
ForwardDiff chunks instead of the single wide chunk it's built to prefer (this package's
own established chunk-width-vs-redundant-re-solve tradeoff — see
[`_pulse_cost_grad_threaded`](../src/pulse_optimizer2.jl)'s own docstring).

Fit-error breakdown (`fit_report`, from
[`_fit_composite_pulse_from_samples_linear`](../src/pulse_optimizer2.jl)):
- **envelope** (`rel_l2_complex`) — full complex `I(t)+iQ(t)` reconstruction error, the
  number that actually reflects what `build_E_of_t(pulse, u_fit)` reproduces.
- **Rabi amplitude** (`rel_l2_A`) — envelope-magnitude-only fit error (`|E(t)|`, the
  taper-weighted amplitude spline fit alone).
- **instantaneous transition frequency** (`rel_l2_f` / `phi_rms_rad`) — pointwise
  instantaneous-frequency fit error, and the RMS error of the *integrated phase* the
  frequency spline actually reconstructs (the quantity that matters for `E(t)`'s phase).


In [ ]:
pulse_fit, u_fit, fit_report, segments = InhomogeneousSpinCavityDynamics.fit_composite_pulse_seed_auto(
    control_cfg, d; fit_mode=:linear, param_budget=60,
)

println("=== Fitted CompositePulse ===")
@printf("  k=%d  n_coeff_A=%d  n_coeff_f=%d  n_params=%d  (%d segments detected)\n",
    pulse_fit.k, pulse_fit.n_coeff_A, pulse_fit.n_coeff_f, n_params(pulse_fit), length(segments))

println()
println("=== Fit error ===")
@printf("  envelope (full I+iQ, rel L2)        = %.6g\n", fit_report.rel_l2_complex)
@printf("  Rabi amplitude (|E|, rel L2)         = %.6g\n", fit_report.rel_l2_A)
@printf("  instantaneous frequency (rel L2)     = %.6g\n", fit_report.rel_l2_f)
@printf("  instantaneous frequency (phase RMS)  = %.6g rad\n", fit_report.phi_rms_rad)
@printf("  amplitude coeffs floored / clipped   = %d / %d\n", fit_report.n_cA_floored, fit_report.n_cf_clipped)


## 6. Dual-track solve on the fitted curve

Runs `:ground` + `:equator` on the **fitted** `CompositePulse` (via
[`pulse_metrics`](../src/pulse_optimizer2.jl)), with the recorded signal pulse still
applied as a fixed background drive (`signal_E_of_t`) — directly comparable to steps 3-4's
numbers for the original recorded pulse.


In [ ]:
signal_E_fit = build_E_of_t(signal_cfg)
inversion_fit, silencing_fit, coherence_fit = pulse_metrics(
    u_fit, pulse_fit, d; signal_E_of_t=signal_E_fit, reltol=RELTOL, abstol=ABSTOL, compute=:cpu,
)

println("=== Fitted pulse vs recorded pulse ===")
@printf("  %-10s  %10s  %10s\n", "metric", "recorded", "fitted")
@printf("  %-10s  %10.6g  %10.6g\n", "inversion", inversion_ref, inversion_fit)
@printf("  %-10s  %10.6g  %10.6g\n", "silencing", silencing_ref, silencing_fit)
@printf("  %-10s  %10.6g  %10.6g\n", "coherence", coherence_ref, coherence_fit)


## 7. Pulse optimisation — single `k`, single seed, no basin-hopping

Warm-starts [`optimise_composite_pulse`](../src/pulse_optimizer2.jl) directly from the
step-5 fit (`warm_start_u=u_fit`), fixes `k`/`n_coeff_A`/`n_coeff_f`/`degree`/`taper_frac`
to that exact fitted shape, and sets `n_hops=1` so **no basin-hopping restarts run** —
one `CompositePulse`, one seed (the fit itself, not a canonical HS1/CORPSE/BB1 multi-seed
sweep), one local Adam descent. This is deliberately the single-k, single-seed subset of
what [`optimise_composite_pulse_over_k`](../src/pulse_optimizer2.jl) would otherwise sweep.

`compute=:cpu` is explicit (not `:auto`) — this machine has a known GPU-driver crash risk
under sustained CUDA compute; drop to `:auto`/`:gpu` only if you've confirmed the GPU path
is safe to use right now. `NUM_EPOCHS` is kept small below (this is a real `M=$(d.M)`
ensemble — each epoch differentiates through a full 1st-order solve); raise it for an
actual search.

**`GRAD_MODE`** picks the gradient backend (forwarded to [`run_local_adam`](../src/pulse_optimizer2.jl)
inside [`optimise_composite_pulse`](../src/pulse_optimizer2.jl)):

- `:forwarddiff` (default here) — the production Dual-Tsit5 ODE gradient. Proven safe at
  large `M` but slow (no memory-scaling issue, since ForwardDiff never materialises a full
  per-step trajectory — the Dual solve differentiates through one adaptive pass directly).
- `:adjoint` — [`pulse_cost_grad_adjoint`](../src/pulse_adjoint.jl)'s frozen-mesh discrete
  Tsit5 adjoint (CPU-only). Can be substantially faster than `:forwarddiff` (no per-parameter
  Dual-number ODE solve), but **at a large ensemble it MUST be paired with
  `USE_CHECKPOINTS=true` and an explicit, finite `CHECKPOINT_STRIDE`** — `use_checkpoints=true`
  alone is NOT sufficient: with `checkpoint_stride` left at its default, the reverse sweep
  still replays the entire trajectory as a single window, giving no memory saving at all over
  `use_checkpoints=false`. Leaving `CHECKPOINT_STRIDE` unset while `USE_CHECKPOINTS=true`
  triggers a `@warn` from `pulse_cost_grad_adjoint` for exactly this reason. A stride in the
  low hundreds is a reasonable starting point; it trades reverse-sweep replay overhead against
  peak memory, so there's no single universally-correct value — pick smaller if memory is
  still tight, larger if replay overhead dominates wall-clock time.


In [ ]:
NUM_EPOCHS       = 5
LEARNING_RATE     = 0.05
SEED              = 42
GRAD_MODE         = :forwarddiff  # :forwarddiff (safe, slow) or :adjoint (fast, needs the two settings below at large M)
USE_CHECKPOINTS   = true          # only consulted when GRAD_MODE == :adjoint
CHECKPOINT_STRIDE = 300           # only consulted when GRAD_MODE == :adjoint; REQUIRED alongside USE_CHECKPOINTS=true -- see step 7's own markdown

global_best_u, global_best_cost, opt_pulse, u0, initial_metrics, history, final_metrics, optimizer_settings = optimise_composite_pulse(
    pulse_fit.k, pulse_fit.n_coeff_A, pulse_fit.n_coeff_f, d;
    degree = pulse_fit.degree, taper_frac = pulse_fit.taper_frac,
    warm_start_u = u_fit,
    n_hops = 1,                     # no basin-hopping
    seed = SEED,
    num_epochs = NUM_EPOCHS,
    learning_rate = LEARNING_RATE,
    threaded_grad = true,
    compute = :cpu,
    grad_mode = GRAD_MODE,
    use_checkpoints = USE_CHECKPOINTS,
    checkpoint_stride = CHECKPOINT_STRIDE,
    signal_E_of_t = signal_E_fit,
)

println()
println("=== Optimisation result ===")
println("  initial: cost=$(round(initial_metrics[1]; digits=4))  inversion=$(round(initial_metrics[2]; digits=4))  silencing=$(round(initial_metrics[3]; digits=4))  coherence=$(round(initial_metrics[5]; digits=4))")
println("  final:   cost=$(round(final_metrics[1]; digits=4))  inversion=$(round(final_metrics[2]; digits=4))  silencing=$(round(final_metrics[3]; digits=4))  coherence=$(round(final_metrics[5]; digits=4))")
println("  global_best_cost returned = $(round(global_best_cost; digits=4))")
